In [18]:
import sqlite3
import pandas as pd

In [19]:
# File paths
CSV_FILE = "../../data/processed/processed_data.csv"
DB_FILE = "../../data/database.db"
TABLE_NAME = "records"

In [20]:
# Connect to SQLite
conn = sqlite3.connect(DB_FILE)

In [21]:
# Drop table if already exists
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
conn.commit()

In [22]:
# Read CSV in chunks to handle large files
chunksize = 100_000
for i, chunk in enumerate(pd.read_csv(CSV_FILE, chunksize=chunksize)):
    print(f"Processing chunk {i+1}...")
    chunk.to_sql(TABLE_NAME, conn, if_exists="append", index=False)

# Create index for fast lookup on Customer
cursor = conn.cursor()
cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_customer ON {TABLE_NAME}(Customer)")
cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_merchant ON {TABLE_NAME}(Merchant)")
conn.commit()

Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...


In [23]:
# Close connection
conn.close()
print("Database created successfully.")

Database created successfully.


In [ ]:
def get_last_entry(filters):
    conn = sqlite3.connect(DB_FILE)

    # Build WHERE clause dynamically
    where_clause = " AND ".join([f"{col} = ?" for col in filters.keys()]) if filters else "1=1"
    values = tuple(filters.values())

    query = f"""
        SELECT *
        FROM {TABLE_NAME}
        WHERE {where_clause}
        ORDER BY rowid DESC
        LIMIT 1
    """

    df = pd.read_sql_query(query, conn, params=values)
    conn.close()
    return df

In [36]:
# read latest entry for customer
filters = {"Customer" : 'C1166355595', "Merchant": "M348934600"}
cust_row = get_last_entry(filters)

cust_row

,step,customer,merchant,customer_amount,fraud,customer_prev_step,customer_time_since_last_transaction,customer_transaction_count,customer_amount_sum,customer_avg_amount,...,category_es_home,category_es_hotelservices,category_es_hyper,category_es_leisure,category_es_otherservices,category_es_sportsandtoys,category_es_tech,category_es_transportation,category_es_travel,category_es_wellnessandbeauty
0,179,C1166355595,M348934600,48.47,0,178.0,1.0,177,6428.63,36.319944,...,0,0,0,0,0,0,0,1,0,0
